# RSNA Knee: training notebook

Trains the cross-slice attention classifier (5 folds) on frozen DINOv2 slice features and the LLM report labels.

1. **Inputs**: find the package source, backbone weights and LLM labels.
2. **Features**: reuse precomputed slice features if they are attached, otherwise compute them from the DICOMs.
3. **Labels**: LLM vs gold agreement table, then soft targets, weights and label statistics.
4. **Training**: K-fold training with verbose logs (per-label AUC tables, collapse check).
5. **Results**: out-of-fold AUCs, learning curves and the checkpoints for the submission notebook.

**On Kaggle** attach: the competition data, `knee-src` (package), `knee-weights` (DINOv2), `knee-labels`
(LLM labels JSONL) and, to skip the ~2 h feature computation, the output of the `knee-full-prep` notebook.
Internet can stay off. Everything is written to `/kaggle/working`.

**Locally** the notebook uses the repository layout (`data/`, `weights/`, `runs/`).

## Parameters
Change these, or override any config value in `OVERRIDES` (same syntax as `--set` on the command line).

In [15]:
RUN_NAME = "s224_ctx5_attention"     # folder name under runs/
CONFIG = "configs/full.yaml"          # 448 px preprocessing cache, 224 px DINOv2 input, mirrored features
OVERRIDES = [
    "backbone.image_size=224",        # 448 uses the high-resolution feature variant
    "model.slice_context=5",          # odd int >= 3 or "all"
    "model.head=attention",           # attention | mean
]
RUN_FOLDS = None                      # None = all folds; e.g. [0] for a quick check
EPOCHS = 30
COMPUTE_FEATURES_IF_MISSING = True    # compute features from DICOM when no precomputed set is found

## 1. Inputs

In [17]:
import json
import os
import shutil
import sys
import time
import pandas
import torch
from pathlib import Path

T0 = time.time()
ON_KAGGLE = Path("/kaggle/input").exists()
SKIP_DIRS = ("train_series", "test_series")  # the DICOM trees are huge; never walk them


def find_dir(marker, root="/kaggle/input", max_depth=6):
    """First folder under `root` that contains the file `marker`."""
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_DIRS and depth < max_depth)
        if marker in filenames:
            return Path(dirpath)
    return None


if ON_KAGGLE:
    REPO = find_dir("knee_src.json")
    WEIGHTS = find_dir("knee_weights.json")
    LABELS_IN = find_dir("knee_labels.json")
    WORK = Path("/kaggle/working")
else:
    # cwd, its parents, and their direct subfolders (e.g. a pulled kernel copy beside rsna-knee/); KNEE_REPO overrides
    candidates = [Path(os.environ["KNEE_REPO"])] if "KNEE_REPO" in os.environ else []
    for p in [Path.cwd(), *Path.cwd().parents]:
        candidates += [p, *sorted(c for c in p.iterdir() if c.is_dir() and not c.name.startswith("."))]
    REPO = next((p for p in candidates if (p / "src" / "knee" / "__init__.py").exists() and "build" not in p.parts), None)
    assert REPO is not None, "package source not found: run from inside the repo or set KNEE_REPO"
    WEIGHTS, LABELS_IN, WORK = REPO / "weights", None, REPO
assert REPO is not None, "package source not found: attach the knee-src dataset"
sys.path.insert(0, str(REPO / "src"))
print(f"package: {REPO}\nweights: {WEIGHTS}\nlabels:  {LABELS_IN or 'data/labels (local)'}\noutput:  {WORK}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
from knee.config import load_config
from knee.utils import device_info, get_logger

overrides = list(OVERRIDES) + [f"train.epochs={EPOCHS}", f"train.run_name={RUN_NAME}"]
if RUN_FOLDS is not None:
    overrides.append(f"train.run_folds={list(RUN_FOLDS)}")
if ON_KAGGLE:
    overrides += [f"paths.{key}={WORK / sub}" for key, sub in [
        ("labels_dir", "labels"), ("runs_dir", "runs"), ("features_dir", "features"),
        ("cache_dir", "cache"), ("results_dir", "results")]]
    overrides.append(f"backbone.weights_path={WEIGHTS}")
cfg = load_config(CONFIG, overrides)
log = get_logger("knee.notebook")
print(json.dumps(device_info(), indent=1))
print("data root:", cfg.paths.data_root)

{
 "python": "3.12.3",
 "platform": "Linux-7.0.0-31-generic-x86_64-with-glibc2.39",
 "torch": "not installed"
}
data root: /home/longarm/development/RSNA Knee/rsna-knee/data/raw


## 2. Features
Precomputed features are matched by their tag (backbone, input size, channel mode, patch grid and a hash of the
preprocessing settings), so a mismatching set is never used by accident.

In [ ]:
from knee.features.extract import feature_tag, features_dir, run_prep_embed

tag = feature_tag(cfg)
attached = None
if ON_KAGGLE:
    for dirpath, dirnames, _ in os.walk("/kaggle/input"):
        depth = len(Path(dirpath).relative_to("/kaggle/input").parts)
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and depth < 7]
        if Path(dirpath).name == tag and (Path(dirpath) / "train").is_dir():
            attached = Path(dirpath).parent
            break
if attached is not None:
    cfg.paths.features_dir = str(attached)
fdir = features_dir(cfg, "train")
n_feat = len(list(fdir.glob("*.npz"))) if fdir.exists() else 0
print(f"feature tag {tag}: {n_feat} studies in {fdir}" + (" (attached)" if attached else ""))
if attached is None and COMPUTE_FEATURES_IF_MISSING:
    print("computing missing features from DICOM (resumable; about 1-2 h for all studies on a T4) ...")
    run_prep_embed(cfg, "train", log)
    n_feat = len(list(fdir.glob("*.npz")))
print(f"{n_feat} studies with features")

ModuleNotFoundError: No module named 'torch'

## 3. Labels
The LLM labels are compared with the 58 gold studies (agreement table and kill switch), then turned into soft
targets, weights and a source per study and label. The label statistics are printed before training.

In [ ]:
from knee.labels.build_targets import build_targets
from knee.labels.validate import run_validation

labels_dir = Path(cfg.paths.labels_dir)
labels_dir.mkdir(parents=True, exist_ok=True)
if LABELS_IN is not None:  # the attached dataset is read-only
    shutil.copy(LABELS_IN / cfg.llm.output_jsonl, labels_dir / cfg.llm.output_jsonl)
agreement = run_validation(cfg, log)
targets = build_targets(cfg, log)

## 4. Training

In [ ]:
from knee.train.train import run_training

run_dir = run_training(cfg)

## 5. Results

In [ ]:
import pandas as pd

metrics = pd.read_csv(run_dir / "metrics.csv")
best = metrics.loc[metrics.groupby("fold").macro_auc_llm.idxmax(),
                   ["fold", "epoch", "macro_auc_llm", "macro_auc_gold", "val_loss"]]
display(best.round(4).reset_index(drop=True))
display(pd.read_csv(run_dir / "oof_auc.csv").round(3))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Fixed categorical order (validated for color-vision deficiency); one panel per measure, no dual axis.
FOLD_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#2b2b29", "#6b6b66", "#e6e6e3"
folds = sorted(metrics.fold.unique())
if len(folds) <= len(FOLD_COLORS):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), facecolor=SURFACE)
    panels = [("macro_auc_llm", "Validation macro AUC (LLM labels)"), ("val_loss", "Validation loss")]
    for ax, (col, title) in zip(axes, panels):
        ax.set_facecolor(SURFACE)
        for fold in folds:
            g = metrics[metrics.fold == fold]
            ax.plot(g.epoch, g[col], color=FOLD_COLORS[int(fold)], lw=2, label=f"fold {fold}")
        ax.set_title(title, loc="left", fontsize=11, color=INK)
        ax.set_xlabel("epoch", color=MUTED)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(True, color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#c9c9c4")
        ax.tick_params(colors=MUTED)
    axes[0].legend(frameon=False, fontsize=9, labelcolor=INK)
    plt.tight_layout()
    plt.show()
print(f"artifacts in {run_dir}:", sorted(p.name for p in run_dir.iterdir()))
print(f"notebook time: {(time.time() - T0) / 60:.1f} min")

## Next: submission
Download this notebook's `runs/<RUN_NAME>` output (or train locally), then build the offline submission inputs:

```bash
python scripts/make_kaggle_bundle.py --run runs/<RUN_NAME> --push-bundle --push-kernel
```